# ArXiv API

In [3]:
import urllib.request as libreq
import urllib.parse
import urllib.error

API_base = "https://export.arxiv.org/api/query"

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------
#Modify this section according to your research objectives
output_filename = "arxiv_output.xml" #Name and location of the output XML file

#Select one or more fields in which the keywords will be searched
  #Available fields: (ti  - Title, abd - Abstract, all - All searchable fields, etc.)
  # Example: ["abs"] - Search abstracts only
search_fields = ["abs"]

#Define your research keywords below.
  #Each inner list represents a group of related search terms
  #Terms within a group are connected using the OR operator
  #Individual groups are connected using the group_operator

  #Example:
  #term_groups = [
  #     ["keyword1", "keyword2", "keyword3"],
  #     ["keyword4", "keyword5"],
  #     ["keyword6", "keyword7"]]

  # With the AND group operator, this produces:
  # (keyword1 OR keyword2 OR keyword3)
  # AND
  # (keyword4 OR keyword5)
  # AND
  # (keyword6 OR keyword7)
term_groups = [

]
group_operator = "AND"


#API parameters
start_index = 0 #Index of the first result to retrieve
max_results = 2000 #Maximum number of publications requested in a single API call

#Sorting criterion:
  #relevance      - Relevance to the search query
  #submittedDate  - Original submission date
  #lastUpdatedDate - Most recent update date
sort_by = "submittedDate"
sort_order = "descending" #Sorting order: ascending or descending
request_timeout = 60 #Maximum waiting time for the API response, in seconds
user_agent = "ArxivLiteratureSearch" #Identifier sent to the API


# ------------------------------------------------------------
# FUNCTIONS
# ------------------------------------------------------------
# Validate the user-defined configuration
def validate_config():
    valid_fields = ["abs", "ti", "au", "all", "cat", "co", "jr", "rn"]

    if not search_fields:
        raise ValueError("At least one search field must be specified.")
    for field in search_fields:
        if field not in valid_fields:
            raise ValueError(f"Invalid search field: {field}")

    if not term_groups:
        raise ValueError("No search terms provided. Please configure term_groups.")
    for index, group in enumerate(term_groups):
        if not group:
            raise ValueError(f"Search term group {index + 1} is empty.")
        for term in group:
            if not isinstance(term, str) or not term.strip():
                raise ValueError(f"Invalid search term in group {index + 1}.")

    if group_operator not in ["AND", "OR"]:
        raise ValueError("group_operator must be AND or OR.")

    if start_index < 0:
        raise ValueError("start_index cannot be negative.")
    if max_results <= 0:
        raise ValueError("max_results must be greater than zero.")
    if sort_by not in ["relevance", "submittedDate", "lastUpdatedDate"]:
        raise ValueError(f"Invalid sorting criterion: {sort_by}")
    if sort_order not in ["ascending", "descending"]:
        raise ValueError(f"Invalid sorting order: {sort_order}")

#Build a search expression for a single group of keywords
def build_query_group(terms, fields):
    expressions = []

    for term in terms:
        for field in fields:
            expressions.append(f'{field}:"{term.strip()}"')
    return "(" + " OR ".join(expressions) + ")"

# Build the complete search query
def build_search_query(term_groups, fields, operator):
    queries = [build_query_group(group, fields) for group in term_groups]
    return f" {operator} ".join(queries)

#API URL
def build_api_url(search_query):
    params = {
        "search_query": search_query,
        "start": start_index,
        "max_results": max_results,
        "sortBy": sort_by,
        "sortOrder": sort_order}
    return API_base + "?" + urllib.parse.urlencode(params)

#Send an HTTP request to the arXiv API
def fetch_arxiv_data(url):
    request = libreq.Request(url, headers={"User-Agent": user_agent})

    with libreq.urlopen(request, timeout=request_timeout) as response:
        return response.read()

#Save the raw API response to an XML file
def save_xml(xml_data, filename):
    """Save the raw API response to an XML file."""

    with open(filename, "wb") as f:
        f.write(xml_data)

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main():
    try:
        validate_config()

        search_query = build_search_query(term_groups, search_fields, group_operator)
        url = build_api_url(search_query)

        print("\nSearch query:")
        print(search_query)
        print("\nAPI URL:")
        print(url)

        print("\nRetrieving data from arXiv...")
        xml_data = fetch_arxiv_data(url)
        save_xml(xml_data, output_filename)
        print(f"\nXML data saved to: {output_filename}")

    except ValueError as error:
        print(f"\nConfiguration error: {error}")

    except urllib.error.HTTPError as error:
        print(f"\nHTTP error: {error.code} - {error.reason}")

    except urllib.error.URLError as error:
        print(f"\nConnection error: {error.reason}")

    except TimeoutError:
        print("\nThe API request timed out.")

    except OSError as error:
        print(f"\nFile or system error: {error}")


if __name__ == "__main__":
    main()


Search query:
(abs:"dataset*" OR abs:"data set*" OR abs:"database*" OR abs:"repository" OR abs:"open data*" OR abs:"kinematic*" OR abs:"motion*" OR abs:"trajector*" OR abs:"telemetr*" OR abs:"robot state*" OR abs:"joint position*" OR abs:"joint angle*" OR abs:"joint velocit*" OR abs:"joint coordinate*" OR abs:"joint variable*" OR abs:"joint parameter*") AND (abs:"da Vinci" OR abs:"daVinci" OR abs:"dVSS" OR abs:"dVRK" OR abs:"Intuitive Surgical" OR abs:"RAMIS" OR abs:"RMIS" OR abs:"robot* surgery" OR abs:"robot*-assisted surgery" OR abs:"robot*-assisted minimally invasive surgery" OR abs:"robot* minimally invasive surgery" OR abs:"robot* surgical procedure*" OR abs:"robot* surgical intervention*" OR abs:"surgical robot*")

API URL:
https://export.arxiv.org/api/query?search_query=%28abs%3A%22dataset%2A%22+OR+abs%3A%22data+set%2A%22+OR+abs%3A%22database%2A%22+OR+abs%3A%22repository%22+OR+abs%3A%22open+data%2A%22+OR+abs%3A%22kinematic%2A%22+OR+abs%3A%22motion%2A%22+OR+abs%3A%22trajector%2